# Trabalho Final — Avaliação de Políticas Públicas com Dados (APPD)

**Programa:** Mestrado Profissional em Administração Pública — IDP
**Professor:** Hélio Bomfim de Macêdo Filho
**Integrantes:** [PREENCHER: nomes completos]
**Política avaliada:** Programa Academia da Saúde (Portaria GM/MS nº 719/2011; Portaria GM/MS nº 2.681/2013)

**Pergunta causal:** a implantação do primeiro polo do Programa Academia da Saúde reduziu as internações por doenças cardiovasculares e metabólicas sensíveis à atividade física (doenças hipertensivas, AVC e diabetes) nos municípios que o adotaram?

**Estimando:** ATT — efeito médio sobre os municípios que implantaram ao menos um polo.
**Unidade de análise:** município × ano, 2008–2024.
**Métodos:** escore de propensão (Módulo 03) e diferenças-em-diferenças com estudo de eventos, TWFE e Callaway & Sant'Anna (Módulo 04).

> **Declaração de uso de IA (versão preliminar, a atualizar na Fase 8).** Este caderno foi desenvolvido com apoio do Claude Code (Anthropic), modelo [PREENCHER: nome e versão], para gerar e depurar código. Os autores revisaram cada célula, reexecutaram integralmente o caderno e assumem responsabilidade pelo conteúdo. Prompts e saídas relevantes estão em `docs/ia/`.

**Caderno 02 — análise:** painel, descritiva, escore de propensão, DiD, robustez e síntese.

**Localização do repositório.** Garante que o caderno rode a partir de `notebooks/`, inclusive quando aberto isolado (ex.: Google Colab), caso em que o repositório público é clonado. Todos os caminhos seguintes são relativos à raiz.

In [ ]:
# Localiza a raiz do repositório (pasta com requirements.txt e src/appd).
# Se o caderno foi aberto isolado (ex.: Google Colab), clona o repositório
# público na pasta atual. Depois disso, tudo usa caminhos relativos à raiz.
import os
import subprocess
import sys
from pathlib import Path

URL_REPOSITORIO = "https://github.com/bsousa7/APPD_26.git"


def localizar_raiz():
    for pasta in [Path.cwd(), *Path.cwd().parents]:
        if (pasta / "requirements.txt").exists() and (pasta / "src" / "appd").exists():
            return pasta
    return None


raiz = localizar_raiz()
if raiz is not None and "google.colab" in sys.modules and (raiz / ".git").exists():
    # No Colab, a cópia clonada pode estar desatualizada: atualiza sempre.
    subprocess.run(["git", "-C", str(raiz), "pull", "--ff-only", "-q"], check=True)
if raiz is None:
    copia = Path("APPD_26")
    if (copia / ".git").exists():
        subprocess.run(["git", "-C", str(copia), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", URL_REPOSITORIO, str(copia)], check=True)
    os.chdir(copia)
    if localizar_raiz() is None:
        raise RuntimeError(
            f"A cópia em {Path.cwd()} não tem requirements.txt e src/appd. "
            "Apague a pasta APPD_26 e execute esta célula de novo."
        )
os.chdir(localizar_raiz() / "notebooks")
versao = subprocess.run(["git", "log", "-1", "--format=%h %cs"], capture_output=True, text=True).stdout.strip()
print(f"Pasta de trabalho: {Path.cwd().parent.name}/notebooks | versão do repositório: {versao}")

# O Colab executa as células da aba aberta, não as da cópia do repositório.
# Se o caderno do repositório mudou, esta aba está desatualizada.
import json
import re

NOME_CADERNO = "02_analise.ipynb"
VERSAO_CADERNO = "dc380eb9c1"
copia = Path.cwd() / NOME_CADERNO
if copia.exists():
    celulas = json.loads(copia.read_text(encoding="utf-8"))["cells"]
    fonte = "".join("".join(c["source"]) if isinstance(c["source"], list) else c["source"] for c in celulas)
    no_repositorio = re.search(r'VERSAO_CADERNO = "([0-9a-f]+)"', fonte)
    if no_repositorio and no_repositorio.group(1) != VERSAO_CADERNO:
        raise RuntimeError(
            f"Este caderno (versão {VERSAO_CADERNO}) está desatualizado: a versão no repositório é "
            f"{no_repositorio.group(1)}. Feche esta aba e abra o caderno de novo pelo link do README. "
            "O que já foi extraído para o Drive é reaproveitado.")
print("Versão do caderno:", VERSAO_CADERNO)

## 1. Identificação, pergunta e DAG

### 1.1 Pergunta, estimando e hipóteses

- **Tratamento (D):** ano do primeiro estabelecimento tipo 74 ("Polo Academia da Saúde") no CNES do município; coorte `g` = esse ano, `g = 0` para os nunca tratados.
- **Período e coortes:** painel 2008–2024; pré-tratamento 2008–2012; coortes a partir de 2013. O tipo 74 aparece no CNES de uma vez em dez/2011 (300 municípios em 22 UFs, provável recadastramento de academias preexistentes) e 2012 acrescenta só 8 municípios; os municípios com polo já em 2011 ou 2012 saem da amostra principal (decisão registrada em `docs/decisoes.md`). Amostra do escore de propensão: coortes 2013–2015 contra nunca tratados. Robustez R7: coortes a partir de 2014.
- **Desfecho principal (Y):** internações SUS por doenças hipertensivas, AVC e diabetes (CID-10 I10–I15, I60–I69, E10–E14) por 10 mil habitantes, por município de residência.
- **Desfecho placebo:** internações por apendicite (K35–K38).
- **Estimando:** ATT.
- **Mecanismo:** polo → práticas corporais orientadas → mais atividade física → melhor controle de pressão e glicemia → menos internações. O efeito esperado é **defasado**, o que motiva o estudo de eventos.

### 1.2 DAG

**O que:** codifica as hipóteses causais em um grafo acíclico dirigido.
**Por quê:** justificar quais variáveis entram como controles (confundidores) e quais **não** podem entrar (mediadores, por sobrecontrole; colisores, por viés de seleção).

Arestas (a revisar pelos autores):
- Cada **confundidor** (porte, PIB per capita, cobertura de APS/ESF/NASF, % de idosos, taxa de internação pré-tratamento, leitos SUS, UF/região) afeta a adesão ao programa e as internações.
- **D → Atividade física → Controle de PA e glicemia → Y:** canal principal (mediadores, não controlados).
- **D → Y:** canais diretos não modelados (por exemplo, educação em saúde).
- **D → Atendimentos AB pós ← Y:** colisor; controlar por ele abriria um caminho espúrio.
- **Habilitação de investimento → D:** afeta só o momento da implantação.
- **Porte → Leitos** e **PIB → Cobertura APS:** relações entre confundidores.

In [ ]:
# Célula autossuficiente: a seção 1 vem antes da preparação do ambiente (seção 2).
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd

DIR_FIG = Path("..") / "outputs" / "figuras"
DIR_FIG.mkdir(parents=True, exist_ok=True)

PAPEL = {
    "Polo AS (D)": "tratamento",
    "Internações CV/DM (Y)": "desfecho",
    "Porte populacional": "confundidor",
    "PIB per capita 2010": "confundidor",
    "Cobertura APS/ESF/NASF 2010": "confundidor",
    "% 60+ anos": "confundidor",
    "Taxa pré 2008–2010": "confundidor",
    "Leitos SUS 2010": "confundidor",
    "UF / região": "confundidor",
    "Atividade física": "mediador",
    "Controle PA e glicemia": "mediador",
    "Atendimentos AB pós": "colisor",
    "Habilitação investimento": "so_D",
}
D, Y = "Polo AS (D)", "Internações CV/DM (Y)"
CONFUNDIDORES = [v for v, p in PAPEL.items() if p == "confundidor"]

dag = nx.DiGraph()
dag.add_nodes_from(PAPEL)
dag.add_edges_from((c, D) for c in CONFUNDIDORES)          # afetam a adesão
dag.add_edges_from((c, Y) for c in CONFUNDIDORES)          # afetam as internações
dag.add_edges_from([
    (D, "Atividade física"),                                # mecanismo
    ("Atividade física", "Controle PA e glicemia"),
    ("Controle PA e glicemia", Y),
    (D, Y),                                                 # outros canais (educação em saúde)
    (D, "Atendimentos AB pós"), (Y, "Atendimentos AB pós"), # colisor
    ("Habilitação investimento", D),                        # só afeta o momento de D
    ("Porte populacional", "Leitos SUS 2010"),
    ("PIB per capita 2010", "Cobertura APS/ESF/NASF 2010"),
])
assert nx.is_directed_acyclic_graph(dag)
print(dag.number_of_nodes(), "nós,", dag.number_of_edges(), "arestas")

O grafo é acíclico, condição necessária para aplicar o critério de backdoor.

### 1.3 Teste do critério de backdoor

**O que:** verifica, com `nx.is_d_separator`, se cada conjunto de controles bloqueia todos os caminhos de porta dos fundos entre D e Y sem incluir descendentes de D.
**Por quê:** fundamenta a escolha das covariáveis X do escore de propensão e mostra por que mediador e colisor ficam de fora.

In [ ]:
def satisfaz_backdoor(g: nx.DiGraph, x: str, y: str, z: set) -> bool:
    """Critério de backdoor: Z não contém descendentes de X e d-separa X de Y
    no grafo sem as arestas que saem de X."""
    if z & nx.descendants(g, x):
        return False
    g_bd = g.copy()
    g_bd.remove_edges_from(list(g.out_edges(x)))
    return nx.is_d_separator(g_bd, {x}, {y}, z)

testes = {
    "Confundidores": set(CONFUNDIDORES),
    "Nenhum controle": set(),
    "Confundidores + mediador": set(CONFUNDIDORES) | {"Atividade física"},
    "Confundidores + colisor": set(CONFUNDIDORES) | {"Atendimentos AB pós"},
    "Confundidores sem taxa pré": set(CONFUNDIDORES) - {"Taxa pré 2008–2010"},
}
pd.Series({k: satisfaz_backdoor(dag, D, Y, v) for k, v in testes.items()},
          name="satisfaz backdoor").to_frame()

**Resultado esperado:** apenas o conjunto completo de confundidores satisfaz o critério. Sem controles, ou sem a taxa pré-tratamento, permanecem caminhos abertos; ao incluir o mediador ou o colisor, o conjunto contém descendentes de D e viola o critério. [Os autores devem confirmar a leitura após executar.]

### 1.4 Figura do DAG
Exporta `outputs/figuras/fig01_dag.png`, usada na seção 3 do artigo.

In [ ]:
CORES = {"tratamento": "#2b6cb0", "desfecho": "#c53030", "confundidor": "#4a5568",
         "mediador": "#2f855a", "colisor": "#b7791f", "so_D": "#6b46c1"}
pos = {D: (0, 0), Y: (7, 0), "Atividade física": (2.2, -1.7), "Controle PA e glicemia": (4.8, -1.7),
       "Atendimentos AB pós": (3.5, -4.2), "Habilitação investimento": (-2.2, -2.2)}
for i, c in enumerate(CONFUNDIDORES):
    pos[c] = (-1.5 + i * 1.65, 3.0 + (i % 2) * 1.1)
rotulo = lambda v: v.replace(" ", "\n", 1) if len(v) > 12 else v

fig, ax = plt.subplots(figsize=(14, 8))
nx.draw_networkx_edges(dag, pos, ax=ax, arrowsize=14, edge_color="#a0aec0", width=1.1,
                       min_source_margin=28, min_target_margin=28)
for v, (x, y) in pos.items():
    ax.text(x, y, rotulo(v), ha="center", va="center", fontsize=9, color="white",
            bbox=dict(boxstyle="round,pad=0.45", fc=CORES[PAPEL[v]], ec="none"))
for papel, cor in CORES.items():
    ax.scatter([], [], s=120, marker="s", color=cor, label=papel.replace("so_D", "afeta só D"))
ax.legend(loc="lower right", fontsize=9, frameon=False)
ax.set_xlim(-3.2, 8.3); ax.set_ylim(-5, 4.8)
ax.set_axis_off()
fig.tight_layout()
fig.savefig(DIR_FIG / "fig01_dag.png", dpi=200, bbox_inches="tight")
plt.show()

A figura distingue os papéis das variáveis por cor.

### 1.5 SUTVA

- **Transbordamento:** moradores de municípios vizinhos podem usar o polo, contaminando os controles. Mitigação: desfecho por município de **residência** (`MUNIC_RES`) e, na robustez, exclusão de controles vizinhos de tratados, se houver tempo.
- **Versões do tratamento:** há polos de modalidades diferentes (básica, intermediária, ampliada) e polos "similares". Mitigação: tratamento alternativo pelo incentivo de custeio (81.12/81.13) e discussão no artigo.

## 2. Preparação do ambiente

**O que:** instala as dependências com as versões fixadas em `requirements.txt`, importa as bibliotecas, define caminhos relativos à raiz do repositório e a semente aleatória.
**Por quê:** reprodutibilidade — qualquer pessoa deve conseguir rodar o caderno com "Restart & Run All" a partir de `notebooks/`.

No Colab, a célula pede autorização para montar o Google Drive, onde estão as bases do caderno 01 e onde serão gravadas as figuras e tabelas. A célula também fixa os **parâmetros de desenho**, para que possam ser revistos num único lugar.

In [ ]:
# Escolhe o arquivo de requisitos conforme o ambiente:
# - Google Colab: requirements-colab.txt (só o que o Colab não traz, sem
#   substituir pandas, ipykernel etc., que o próprio Colab exige);
# - ambiente local: requirements.txt (todas as bibliotecas, versões fixadas).
# O caminho é calculado a partir de notebooks/ e passado já resolvido ao %pip.
import sys
from pathlib import Path

NO_COLAB = "google.colab" in sys.modules
nome = "requirements-colab.txt" if NO_COLAB else "requirements.txt"
ARQ_REQUISITOS = (Path.cwd().parent / nome).resolve()
if not ARQ_REQUISITOS.exists():
    raise FileNotFoundError(
        f"{nome} não encontrado a partir de {Path.cwd().name}/. "
        "Execute antes a célula 'Localização do repositório' (seção 1)."
    )
print("Instalando", nome)
%pip install -q -r {ARQ_REQUISITOS}

# Se o pip trocou a versão de uma biblioteca que já estava carregada (comum no
# Colab, que pré-carrega numpy e pandas), a sessão precisa ser reiniciada.
from importlib.metadata import version as versao_instalada

trocadas = [
    f"{nome} {sys.modules[nome].__version__} -> {versao_instalada(nome)}"
    for nome in ["numpy", "pandas", "scipy"]
    if nome in sys.modules and sys.modules[nome].__version__ != versao_instalada(nome)
]
if trocadas:
    raise RuntimeError(
        "Bibliotecas atualizadas: " + "; ".join(trocadas) + ". Reinicie a sessão "
        "(Ambiente de execução > Reiniciar sessão) e execute TUDO de novo, desde a "
        "primeira célula de código. Na segunda execução esta célula passa direto."
    )
print("Dependências prontas.")

Dependências instaladas. Se a célula pedir para reiniciar a sessão, reinicie e execute tudo desde a primeira célula de código.

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

RAIZ = Path("..")
if not (RAIZ / "src" / "appd").exists():
    raise RuntimeError(
        "Pasta de trabalho errada (comum após reiniciar a sessão do Colab). "
        "Execute o caderno desde a primeira célula de código (localização do repositório)."
    )
sys.path.insert(0, str(RAIZ / "src"))
# Descarta versões já carregadas do pacote do projeto: se a primeira célula
# atualizou o código (git pull), a importação abaixo usa a versão nova.
for _m in [m for m in sys.modules if m == "appd" or m.startswith("appd.")]:
    del sys.modules[_m]

DIR_RAW = RAIZ / "data" / "raw"
DIR_PROC = RAIZ / "data" / "processed"
DIR_DIC = DIR_PROC / "dicionarios"
DIR_FIG = RAIZ / "outputs" / "figuras"
DIR_TAB = RAIZ / "outputs" / "tabelas"
for d in (DIR_RAW, DIR_PROC, DIR_DIC, DIR_FIG, DIR_TAB):
    d.mkdir(parents=True, exist_ok=True)

SEMENTE = 42
np.random.seed(SEMENTE)
pd.set_option("display.max_columns", 30)
from importlib.metadata import version
print("Python", sys.version.split()[0], "|", ", ".join(
    f"{p} {version(p)}" for p in ["pandas", "numpy", "scipy", "statsmodels",
                                  "scikit-learn", "linearmodels", "differences", "pysus"]))

from appd import analise as an
from appd import estimacao as est
from appd.armazenamento import preparar_armazenamento, preparar_saidas

# No Colab, data/processed/ e outputs/ ficam no Google Drive (APPD_26_dados).
DIR_PROC = preparar_armazenamento(RAIZ)
DIR_SAIDAS = preparar_saidas(RAIZ)
DIR_FIG = DIR_SAIDAS / "figuras"
DIR_TAB = DIR_SAIDAS / "tabelas"

# ------------------------------------------- parâmetros de desenho (docs/decisoes.md)
ANO_PRIMEIRA_COORTE = 2013      # municípios com polo antes disso saem da amostra
TRATAMENTO_ABSORVENTE = True    # D = 1 a partir do primeiro polo (intenção de tratar)
COORTES_PSM = (2013, 2015)      # tratados da amostra do escore de propensão
JANELA_PRE = (2008, 2010)       # taxa pré-tratamento
JANELA_POS = (2016, 2018)       # desfecho do PSM: variação pós − pré
TIPOS_EQUIPE_APS = None         # PROVISÓRIO: None = todos os tipos de equipe do CNES-EP

# ------------------------------------------------ estilo dos gráficos (paleta validada)
CORES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]
COR_NEUTRA, TINTA, TINTA_2, GRADE, EIXO = "#b5b3ad", "#0b0b0b", "#52514e", "#e1e0d9", "#c3c2b7"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "axes.edgecolor": EIXO, "axes.labelcolor": TINTA_2, "xtick.color": TINTA_2, "ytick.color": TINTA_2,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "axes.grid.axis": "y",
    "grid.color": GRADE, "grid.linewidth": 0.6, "font.family": "DejaVu Sans", "font.size": 10, "axes.titlesize": 11,
    "axes.titleweight": "bold", "axes.titlecolor": TINTA, "legend.frameon": False,
})

Ambiente pronto. Os parâmetros de desenho seguem `docs/decisoes.md`:
- coortes a partir de **2013** (municípios com polo em 2011–2012 saem da amostra principal);
- tratamento **absorvente**: D = 1 a partir do primeiro polo, mesmo que o polo deixe de constar no CNES (cerca de 280 municípios);
- `TIPOS_EQUIPE_APS = None` é **provisório**: soma todos os tipos de equipe do CNES-EP até que os autores confiram os rótulos de `TIPO_EQP`.

## 3. Painel e validação

### 3.1 Leitura das bases do caderno 01
**O que:** lê o painel município × ano e as covariáveis de 2010 gravados pelo `01_extracao.ipynb`.
**Por quê:** a análise parte só das bases geradas por código; identificadores são lidos como texto.

In [ ]:
painel_bruto = pd.read_parquet(DIR_PROC / "painel_bruto.parquet")
painel_bruto["cod6"] = painel_bruto["cod6"].astype(str)
covariaveis = pd.read_csv(DIR_PROC / "covariaveis_2010.csv", dtype={"cod6": str, "cod_ibge7": str})

assert not painel_bruto.duplicated(["cod6", "ano"]).any()
assert covariaveis["cod6"].is_unique
ANOS = sorted(painel_bruto["ano"].unique())
print(f"Painel: {len(painel_bruto):,} linhas | {painel_bruto['cod6'].nunique():,} municípios | "
      f"anos {ANOS[0]}–{ANOS[-1]}")
print(f"Covariáveis: {len(covariaveis):,} municípios")

O painel tem uma linha por município e ano, sem duplicatas; as covariáveis têm uma linha por município.

### 3.2 Taxas e tratamento
**O que:** calcula as taxas por 10 mil habitantes (desfecho principal: hipertensão + AVC + diabetes; secundários: cada grupo; placebo: apendicite) e define a coorte `g` (primeiro ano com polo tipo 74; `g = 0` para nunca tratados) e o indicador `D`.
**Por quê:** `D` é a variável de tratamento dos métodos; com `TRATAMENTO_ABSORVENTE = True`, um município continua tratado depois do primeiro polo, mesmo que o polo deixe de constar no CNES.

In [ ]:
painel = an.definir_tratamento(an.calcular_taxas(painel_bruto), absorvente=TRATAMENTO_ABSORVENTE)
unidades = painel.drop_duplicates("cod6")
print(f"Municípios com polo em algum ano: {(unidades['g'] > 0).sum():,} | nunca tratados: {(unidades['g'] == 0).sum():,}")
print(f"Municípios que deixaram de ter polo após o primeiro: {unidades['perdeu_polo'].sum():,}")
painel[["taxa_cv_dm", "taxa_hipertensao", "taxa_avc", "taxa_diabetes", "taxa_apendicite"]].describe().round(2)

**Resultado:** [PREENCHER: número de tratados, nunca tratados e municípios que perderam o polo; faixas das taxas]

### 3.3 Amostra de diferenças-em-diferenças
**O que:** aplica as exclusões, com a contagem de cada passo:
1. municípios sem população em algum ano (instalados durante o período, sem série completa);
2. municípios com polo antes de `ANO_PRIMEIRA_COORTE` (provável recadastramento de academias preexistentes).

**Por quê:** o DiD exige painel balanceado e coortes cuja data de tratamento seja confiável. As validações verificam painel balanceado, chave única, ausência de faltantes no desfecho e em `D` e, com tratamento absorvente, `D` que nunca volta a zero.

In [ ]:
painel_did, exclusoes = an.selecionar_amostra_did(painel, ANO_PRIMEIRA_COORTE)
n_mun = painel_did["cod6"].nunique()
assert len(painel_did) == n_mun * len(ANOS), "painel desbalanceado"
assert not painel_did.duplicated(["cod6", "ano"]).any()
assert painel_did[["taxa_cv_dm", "taxa_apendicite", "D", "g"]].notna().all().all()
assert (painel_did["populacao"] > 0).all()
if TRATAMENTO_ABSORVENTE:
    assert (painel_did.sort_values(["cod6", "ano"]).groupby("cod6")["D"].diff().fillna(0) >= 0).all()
assert painel_did.loc[painel_did["g"] > 0, "g"].min() >= ANO_PRIMEIRA_COORTE

est.exportar_tabela(exclusoes, DIR_TAB, "tabA1_exclusoes", decimais=0)
print(f"Amostra DiD: {n_mun:,} municípios × {len(ANOS)} anos = {len(painel_did):,} linhas")
exclusoes

**Resultado:** [PREENCHER: municípios excluídos em cada passo e tamanho final da amostra DiD]

### 3.4 Base do escore de propensão
**O que:** base transversal com um município por linha: tratados das coortes `COORTES_PSM` (2013–2015) e nunca tratados; covariáveis de 2010 (porte, PIB per capita, % de 60+, leitos SUS por mil, equipes por 10 mil, região) e taxa pré-tratamento (média de 2008–2010).
**Desfecho do PSM:** variação da taxa entre a média de `JANELA_POS` (2016–2018) e a de `JANELA_PRE` (2008–2010), o "PSM-DiD" do PLANO.md, seção 4.4.
**Por quê:** as covariáveis seguem o DAG (seção 1); mediadores e colisores não entram.

In [ ]:
base_psm = an.montar_base_psm(painel_did, covariaveis, coortes_tratadas=COORTES_PSM,
                              janela_pre=JANELA_PRE, janela_pos=JANELA_POS,
                              tipos_equipe_aps=TIPOS_EQUIPE_APS)
COVARIAVEIS_X = ["log_pop", "log_pib_per_capita", "pct_60_mais", "leitos_sus_por_mil",
                 "equipes_aps_por_10mil", "taxa_pre"]
assert base_psm["cod6"].is_unique
assert base_psm[COVARIAVEIS_X + ["delta_taxa", "regiao"]].notna().all().all(), \
    base_psm[COVARIAVEIS_X + ["delta_taxa"]].isna().sum()

painel_did.to_parquet(DIR_PROC / "painel_analitico.parquet", index=False)
base_psm.to_parquet(DIR_PROC / "base_psm.parquet", index=False)
print(base_psm.groupby("tratado").size().rename({0: "nunca tratados", 1: "tratados 2013–2015"}).to_string())

**Resultado:** [PREENCHER: número de tratados (coortes 2013–2015) e de controles; nenhuma covariável faltante]

## 4. Análise descritiva

**Atenção:** esta seção descreve os grupos e as tendências. Diferenças entre tratados e nunca tratados aqui **não** são efeitos do programa: os grupos diferem em características que também afetam as internações (DAG, seção 1). A leitura causal fica para as seções 5 e 6.

### 4.1 Tabela 1 — características pré-tratamento por grupo
**O que:** médias e desvios-padrão das covariáveis de 2010 e das taxas de 2008–2010, para os municípios tratados (todas as coortes a partir de 2013) e os nunca tratados da amostra DiD.

In [ ]:
base_desc = an.montar_base_psm(painel_did, covariaveis, coortes_tratadas=(ANO_PRIMEIRA_COORTE, ANOS[-1]),
                               janela_pre=JANELA_PRE, janela_pos=JANELA_POS,
                               tipos_equipe_aps=TIPOS_EQUIPE_APS)
for g_ in ["hipertensao", "avc", "diabetes", "apendicite"]:
    base_desc = base_desc.merge(an.media_periodo(painel_did, f"taxa_{g_}", *JANELA_PRE).rename(f"pre_{g_}"),
                                left_on="cod6", right_index=True)
VARIAVEIS_TAB1 = {
    "pop_total_censo": "População (Censo 2010)",
    "pib_per_capita": "PIB per capita 2010 (R$)",
    "pct_60_mais": "População com 60 anos ou mais (%)",
    "leitos_sus_por_mil": "Leitos SUS por mil hab. (2010)",
    "equipes_aps_por_10mil": "Equipes de saúde por 10 mil hab. (2010)",
    "taxa_pre": "Internações CV/DM por 10 mil hab. (2008–2010)",
    "pre_hipertensao": "  Doenças hipertensivas",
    "pre_avc": "  AVC",
    "pre_diabetes": "  Diabetes",
    "pre_apendicite": "Internações por apendicite por 10 mil hab. (2008–2010)",
}
tab01 = an.tabela_descritiva(base_desc, VARIAVEIS_TAB1)
est.exportar_tabela(tab01, DIR_TAB, "tab01_descritiva")
tab01.round(2)

**Resultado:** [PREENCHER: descrição das diferenças entre os grupos, sem interpretação causal]

### 4.2 Figura 2 — municípios por coorte de adoção
**O que:** número de municípios por ano do primeiro polo tipo 74. As coortes de 2011–2012 aparecem em cinza porque saem da amostra principal.

In [ ]:
contagem = painel.drop_duplicates("cod6").query("g > 0")["g"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(8, 3.6))
cores_barras = [COR_NEUTRA if ano < ANO_PRIMEIRA_COORTE else CORES[0] for ano in contagem.index]
ax.bar(contagem.index, contagem.values, color=cores_barras, width=0.7, edgecolor="#fcfcfb", linewidth=2)
ax.set_xticks(contagem.index)
ax.set_xlabel("Ano do primeiro polo tipo 74 no CNES (coorte g)")
ax.set_ylabel("Municípios")
nunca = (painel.drop_duplicates("cod6")["g"] == 0).sum()
ax.set_title("Municípios por ano do primeiro polo do Programa Academia da Saúde", loc="left")
ax.text(0.99, 0.95, f"Nunca tratados: {nunca:,}".replace(",", "."), transform=ax.transAxes,
        ha="right", va="top", color=TINTA_2, fontsize=9)
ax.text(contagem.index[0] + 0.5, contagem.max() * 0.92, "excluídas da\namostra principal",
        color=TINTA_2, fontsize=9, ha="center")
fig.tight_layout()
fig.savefig(DIR_FIG / "fig02_coortes.png", dpi=200)
plt.show()
contagem.rename("municípios").to_frame().T

**Resultado:** [PREENCHER: distribuição das coortes; concentração da adoção em 2014–2016]

### 4.3 Figura 3 — evolução do desfecho por grupo de coorte
**O que:** média municipal da taxa de internações por hipertensão, AVC e diabetes (por 10 mil hab.), por ano, para grupos de coorte da amostra DiD. A tabela abaixo do gráfico traz os mesmos valores.
**Por quê:** primeira inspeção visual das tendências antes do tratamento; o exame formal das tendências paralelas é o estudo de eventos da seção 6.

In [ ]:
painel_did["grupo_coorte"] = an.grupo_de_coorte(painel_did["g"])
ORDEM = ["2013", "2014", "2015", "2016–2018", "2019–2024", "Nunca tratados"]
tendencias = (painel_did.groupby(["grupo_coorte", "ano"])["taxa_cv_dm"].mean()
              .unstack("grupo_coorte")[ORDEM])
fig, ax = plt.subplots(figsize=(8, 4.6))
for cor, grupo in zip(CORES, ORDEM):
    ax.plot(tendencias.index, tendencias[grupo], color=cor, linewidth=2, label=grupo)
ax.set_xlim(tendencias.index[0], tendencias.index[-1])
ax.set_xticks(tendencias.index[::2])
ax.set_ylabel("Internações por 10 mil hab.")
ax.set_xlabel("Ano")
ax.set_title("Internações por doenças hipertensivas, AVC e diabetes, por grupo de coorte", loc="left")
# 6 séries: legenda fora da área do gráfico (rótulos no fim das linhas se sobrepõem)
ax.legend(title="Coorte (ano do 1º polo)", fontsize=8.5, title_fontsize=8.5, ncol=6,
          loc="upper center", bbox_to_anchor=(0.5, -0.17), handlelength=1.6, columnspacing=1.2)
fig.tight_layout()
fig.savefig(DIR_FIG / "fig03_tendencias.png", dpi=200)
plt.show()
est.exportar_tabela(tendencias.reset_index(), DIR_TAB, "tabA2_tendencias")
tendencias.round(1)

**Resultado:** [PREENCHER: padrão das séries antes de 2013 e evolução no período, sem leitura causal]

## 5. Método 1 — escore de propensão
*Fase 5:* hipóteses de ignorabilidade e suporte comum; logit e gradient boosting calibrado; `fig04_suporte_comum`; vizinho mais próximo com caliper e IPW (pesos ATT); `tab02_balanceamento` e `fig05_loveplot`; ATT com bootstrap (1.000 réplicas, `SEMENTE`).

## 6. Método 2 — diferenças-em-diferenças
*Fase 6:* tendências paralelas e não antecipação; estudo de eventos (`fig06`); TWFE clusterizado (`tab03`); Goodman-Bacon (`fig07`); Callaway & Sant'Anna (`fig08`, `tab04`); placebos (`tab05`).

## 7. Robustez e poder
*Fase 6:* R1–R7 (`tab06_robustez`) e MDE (`tab07_mde`).

## 8. Síntese e exportação
*Fase 6:* `tab08_sintese` (estimativa, EP, IC 95%, N e nota de cada método), exportada em `.csv` e `.tex` com o mesmo nome-base usado no artigo.